# cli

> drive the vault from a terminal

In [ ]:
#| default_exp cli

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import json, os, sys
from fastcore.script import call_parse, anno_parser
from vishalakshi.core import Vault

def _v(path=None, offline=False):
    'Open the vault the CLI operates on ($VISHALAKSHI_VAULT, else the default path).'
    return Vault(path or os.getenv('VISHALAKSHI_VAULT') or None,
                 offline=offline or bool(os.getenv('VISHALAKSHI_OFFLINE')))

def _out(obj, as_json):
    if as_json: print(json.dumps(obj, default=str)); return True
    return False

In [ ]:
#| export
@call_parse
def status(vault:str=None, as_json:bool=False):
    "What is in the vault: counts by kind, and which encoder is active."
    s = _v(vault).stats()
    if _out(s, as_json): return
    print(f"{s['path']}\n  {s['docs']} docs · {s['nodes']} sections · {s['chunks']} chunks · "
          f"{s['entities']} entities\n  encoder: {s['encoder']}")
    for k, n in sorted(s['by_kind'].items(), key=lambda kv: -kv[1]): print(f"  {n:5} {k}")

@call_parse
def search(query:str, n:int=10, kind:str=None, vault:str=None, as_json:bool=False):
    "Find chunks across the vault (keyword + vector, fused)."
    ks = kind.split(',') if kind else None
    hits = _v(vault).find(query, limit=n, kind=ks)
    if _out(hits, as_json): return
    for h in hits:
        print(f"## {h.get('breadcrumb')}")
        print(f"   {h.get('node_id')}")
        print(f"   {(h.get('content') or '')[:300]}\n")

@call_parse
def context(question:str, sections:int=6, related:int=6, kind:str=None, chars:int=1200,
            vault:str=None, as_json:bool=False):
    "The sections worth reading for a question, plus what they connect to."
    c = _v(vault).context(question, sections=sections, related=related,
                          kind=kind.split(',') if kind else None)
    if as_json:
        print(json.dumps({'question': question, 'encoder': c.encoder,
                          'results': [dict(r, tree=None, snippets=list(r.snippets)) for r in c.results],
                          'related': [dict(r) for r in c.related]}, default=str)); return
    print(f"# {question}\n({c.encoder})\n")
    for i, r in enumerate(c.results, 1):
        print(f"[{i}] {r.breadcrumb}\n    source: {r.filename}  pages: {r.pages}")
        print(f"    {(r.text or '')[:chars]}\n")
    if c.related:
        print('RELATED')
        for r in c.related: print(f"  - {r.breadcrumb}  (via {r.via})")

@call_parse
def ask(question:str, model:str=None, sections:int=6, kind:str=None, vault:str=None, as_json:bool=False):
    "Answer a question from the vault, with citations back into it."
    r = _v(vault).ask(question, model=model, sections=sections,
                      kind=kind.split(',') if kind else None)
    if _out({'question': r.question, 'answer': r.answer, 'model': r.model, 'cited': r.cited}, as_json): return
    print(r.answer)
    if r.cited:
        print('\nSources')
        for c in r.cited: print(f"  [{c['n']}] {c['breadcrumb']}  ({c['source']})  {c['node_id']}")

@call_parse
def read(node_id:str, chars:int=8000, vault:str=None, as_json:bool=False):
    "Read one section of the vault in full."
    s = _v(vault).read(node_id, max_chars=chars)
    if _out(s, as_json): return
    print(f"# {s.get('title','')}\n\n{s.get('text','')}")

@call_parse
def related(node_id:str, n:int=8, vault:str=None, as_json:bool=False):
    "What else in the vault reads like this section."
    rs = _v(vault).related(node_id, limit=n)
    if _out(rs, as_json): return
    for r in rs: print(f"{r['breadcrumb']}\n  {r['node_id']}\n  {r['snippet'][:160]}\n")

@call_parse
def toc(vault:str=None, as_json:bool=False):
    "The table of contents across every document in the vault."
    t = _v(vault).toc()
    if _out(t, as_json): return
    def walk(nd, d=0):
        print('  '*d + f"{nd['title']}  [{nd['id']}]")
        for c in nd.get('children', []): walk(c, d+1)
    for doc in t:
        print(f"\n{doc['title']}  ({doc['source']})")
        walk(doc['tree'], 1)

@call_parse
def topics(min_count:int=2, vault:str=None, as_json:bool=False):
    "Cluster the vault into labelled topics."
    c = _v(vault).map(min_count=min_count)
    if _out({'method': c.method, 'note': c.note,
             'clusters': [{'label': x.label, 'size': x.size} for x in c.clusters]}, as_json): return
    print(c.note)
    for x in c.clusters: print(f"  [{x.size:4}] {x.label}")

@call_parse
def sources(kind:str=None, vault:str=None, as_json:bool=False):
    "Every document in the vault and where it came from."
    ss = _v(vault).sources(kind=kind.split(',') if kind else None)
    if _out(ss, as_json): return
    for s in ss:
        q = s['meta'].get('query')
        print(f"[{s['kind']:7}] {s['title'][:60]}\n          {s['source']}"
              + (f"\n          found by: {q!r}" if q else '') + f"\n          {s['doc_id']}")

In [ ]:
#| export
@call_parse
def add(target:str, title:str=None, kind:str=None, sel:str=None, vault:str=None, as_json:bool=False):
    """Put something in the vault: a URL, an arXiv id, a YouTube link, a file or a directory.

    The kind is inferred from the target unless you pass --kind."""
    v, t = _v(vault), target
    if os.path.isdir(t):                                    r = v.add_dir(t)
    elif os.path.exists(t):                                 r = v.add_file(t, title=title, kind=kind)
    elif 'arxiv.org' in t or __import__('re').fullmatch(r'\d{4}\.\d{4,5}(v\d+)?', t): r = v.arxiv(t)
    elif 'youtube.com' in t or 'youtu.be' in t:             r = v.youtube(t)
    elif t.startswith('http'):                              r = v.url(t, title=title, sel=sel)
    else: print(f'not a URL, file or directory: {t}', file=sys.stderr); sys.exit(1)
    if _out(r, as_json): return
    print(json.dumps(r, indent=2, default=str))

@call_parse
def web(query:str, n:int=5, google:bool=False, vault:str=None, as_json:bool=False):
    "Search the web, read the top results, and file them in the vault."
    r = _v(vault).web(query, n=n, google=google)
    if _out(r, as_json): return
    print(f"{query!r}: {r['n_found']} found, {len(r['added'])} readable")
    for a in r['added']:
        print(f"  {'skipped' if a.get('skipped') else 'added':8} {a.get('title','')[:60]}  {a.get('url','')}")

@call_parse
def note(text:str, title:str=None, tags:str=None, vault:str=None, as_json:bool=False):
    "Write a note into the vault so it is searched alongside the sources."
    r = _v(vault).note(text, title=title, tags=tags.split(',') if tags else None)
    if _out(r, as_json): return
    print(json.dumps(r, indent=2, default=str))

@call_parse
def connect(vault:str=None, as_json:bool=False):
    "(Re)build the entity graph over the vault — enables the associative retrieval leg."
    r = {k: x for k, x in _v(vault).connect().items() if k != 'resolved'}
    if _out(r, as_json): return
    print(json.dumps(r, indent=2, default=str))

@call_parse
def forget(doc_id:str, vault:str=None):
    "Remove a document, its sections and its chunks from the vault."
    _v(vault).forget(doc_id)
    print(f'forgot {doc_id}')

In [ ]:
#| export
@call_parse
def models(as_json:bool=False):
    "Model aliases and which backend each one runs on."
    from .ask import MODELS, dflt_model
    if _out({'default': dflt_model, 'models': {k: dict(id=v[0], runtime=v[1], note=v[2])
                                               for k, v in MODELS.items()}}, as_json): return
    print(f'default: {dflt_model}\n')
    for k, (mid, rt, nt) in MODELS.items(): print(f'  {k:15} {rt:7} {mid:45} {nt}')

@call_parse
def index_code(dir:str=None, graph:bool=True, env:bool=False, force:bool=False, vault:str=None):
    "Point the vault at a repo and fill kosha's code store and call graph."
    print(json.dumps(_v(vault).index_code(dir or '.', graph=graph, env=env, force=force),
                     indent=2, default=str))

@call_parse
def code(query:str, n:int=10, env:bool=True, dir:str=None, vault:str=None, as_json:bool=False):
    "Search code through kosha (supports key:value filters like package:httpx)."
    rows = _v(vault).code_search(query, limit=n, env=env, dir=dir)
    out = [dict(mod=r.get('metadata',{}).get('mod_name'), path=r.get('metadata',{}).get('path'),
                lineno=r.get('metadata',{}).get('lineno')) for r in rows]
    if _out(out, as_json): return
    for r in out: print(f"{r['mod']}\n  {r['path']}:{r['lineno']}")

@call_parse
def symbol(name:str, depth:int=1, dir:str=None, vault:str=None, as_json:bool=False):
    "A symbol in the call graph: pagerank, degree, callers and callees."
    s = _v(vault).symbol(name, depth=depth, dir=dir)
    if _out(dict(node=s.node, info=s.info, callers=list(s.callers), callees=list(s.callees)), as_json): return
    print(f"{s.node}\n  pagerank: {s.info.get('pagerank')}  in/out: {s.info.get('in_degree')}/{s.info.get('out_degree')}")
    print(f"  callers: {list(s.callers)[:10]}\n  callees: {list(s.callees)[:10]}")

@call_parse
def federate(query:str, n:int=12, prose:bool=True, repo:bool=True, env:bool=False,
             dir:str=None, vault:str=None, as_json:bool=False):
    "One ranked list across your documents and your code."
    from .code import fed_rows
    f = _v(vault).federate(query, limit=n, prose=prose, repo=repo, env=env, dir=dir)
    if _out({'query': query, 'legs': f.legs, 'note': f.note, 'hits': fed_rows(f.hits)}, as_json): return
    print(f"{f.note}\nlegs: {f.legs}\n")
    for i, h in enumerate(f.hits, 1): print(f"[{i}] ({h.source}) {h.where}\n    {h.text[:160]}\n")

In [ ]:
#| export
@call_parse
def apis(url:str, pattern:str='*', session:bool=False, vault:str=None, as_json:bool=False):
    "Discover the JSON endpoints a page calls, so you can read its data instead of its HTML."
    rows = _v(vault).apis(url, pattern=pattern, session=session)
    if _out([dict(r) for r in rows], as_json): return
    for r in rows: print(f"[{r.n}] {r.url}\n    records: {r.records}  type: {r.content_type}\n    {r.preview[:200]}\n")

@call_parse
def harvest(url:str, pattern:str='*', capture:int=None, pages:int=1, title:str=None,
            session:bool=False, force:bool=False, vault:str=None, as_json:bool=False):
    "Sniff a page's JSON API, pull the records, and file them in the vault as kind='data'."
    r = _v(vault).harvest(url, pattern=pattern, capture=capture, pages=pages, title=title,
                          session=session, force=force)
    if _out(r, as_json): return
    print(json.dumps(r, indent=2, default=str))

In [ ]:
#| export
@call_parse
def watch(target:str, action:str='url', every:str='1d', note:str=None, vault:str=None, as_json:bool=False):
    "Register a recurring job: re-read a page, re-run a search, re-harvest an API, or remind you."
    r = _v(vault).watch(target, action=action, every=every, note=note)
    if _out(r, as_json): return
    print(json.dumps(r, indent=2, default=str))

@call_parse
def watches(vault:str=None, as_json:bool=False):
    "Every registered watch, soonest first."
    ws = _v(vault).watches()
    if _out(ws, as_json): return
    import time as _t
    for w in ws:
        due = (w['next_run'] or 0) - _t.time()
        print(f"{w['id']}  {w['action']:8} every {w['every']:4} "
              f"{'DUE' if due <= 0 else f'in {due/3600:.1f}h':>10}  runs={w['runs']} "
              f"{w['last_status'] or '-'}\n    {w['target'][:70]}")

@call_parse
def poll(limit:int=None, vault:str=None, as_json:bool=False):
    "Run every watch that is due. This is the tick a cron or a frontend calls."
    r = _v(vault).poll(limit=limit)
    if _out(r, as_json): return
    print(f"checked {r['checked']}, ran {r['ran']}")
    for x in r['results']: print(f"  {x['action']:8} {x['status']:8} {x['took']}s  {x['target'][:50]}")

@call_parse
def unwatch(watch_id:str, vault:str=None):
    "Delete a watch. Documents it already filed stay in the vault."
    _v(vault).unwatch(watch_id); print(f'unwatched {watch_id}')

In [ ]:
#| export
CMDS = {'status': status, 'search': search, 'context': context, 'ask': ask, 'read': read,
        'related': related, 'toc': toc, 'topics': topics, 'sources': sources,
        'add': add, 'web': web, 'note': note, 'connect': connect, 'forget': forget,
        'models': models, 'index-code': index_code, 'code': code, 'symbol': symbol,
        'federate': federate, 'apis': apis, 'harvest': harvest,
        'watch': watch, 'watches': watches, 'poll': poll, 'unwatch': unwatch}

def main():
    "Entry point for the `vishalakshi` CLI command."
    if len(sys.argv) < 2 or sys.argv[1] not in CMDS:
        print(f"Usage: vishalakshi [{' | '.join(CMDS)}]")
        sys.exit(0 if len(sys.argv) < 2 else 1)
    func = CMDS[sys.argv.pop(1)]
    raw = getattr(func, '__wrapped__', func)
    args = anno_parser(raw).parse_args().__dict__
    args.pop('xtra', None); args.pop('pdb', None)
    raw(**args)